# 🔍 RAG Embedding Model — From Scratch (Colab)

**Bi-Encoder** 구조로 임베딩 모델을 처음부터 학습합니다.

| 구성 요소 | 설명 |
|---|---|
| 인코더 | Transformer (Pre-LN, 6 layers, 8 heads) |
| 풀링 | Mean Pooling + L2 정규화 |
| 손실 함수 | InfoNCE (in-batch negatives) |
| 데이터 | 합성 데이터 또는 MS MARCO |
| 평가 | MRR@10, Recall@{1,5,10} |

### ▶ 실행 순서
1. 런타임 → **GPU로 변경** (T4 무료)
2. 셀을 위에서 아래로 순서대로 실행

In [ ]:
import torch
print('PyTorch version :', torch.__version__)
print('GPU available   :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU name        :', torch.cuda.get_device_name(0))
    print('VRAM            :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('GPU 없음. 런타임 → 런타임 유형 변경 → T4 GPU 선택 후 재실행하세요.')


## 1. 설정

In [ ]:
import math, random
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.notebook import tqdm
from dataclasses import dataclass

@dataclass
class ModelConfig:
    vocab_size : int   = 30_522
    max_seq_len: int   = 128
    d_model    : int   = 512
    n_heads    : int   = 8
    n_layers   : int   = 6
    d_ff       : int   = 2048
    dropout    : float = 0.1
    embed_dim  : int   = 256

@dataclass
class TrainConfig:
    batch_size   : int   = 64
    learning_rate: float = 2e-4
    warmup_steps : int   = 500
    epochs       : int   = 5
    temperature  : float = 0.05
    max_grad_norm: float = 1.0
    eval_every   : int   = 200
    save_path    : str   = '/content/embedding_model.pt'
    device       : str   = 'cuda' if torch.cuda.is_available() else 'cpu'

# 데이터 설정
USE_SYNTHETIC   = True    # False → MS MARCO (datasets 패키지 필요)
SYNTHETIC_N     = 8_000
MSMARCO_SAMPLES = 50_000

model_cfg = ModelConfig()
train_cfg = TrainConfig()
print(f'Device: {train_cfg.device}')


## 2. 토크나이저

In [ ]:
class SimpleTokenizer:
    PAD, UNK, CLS, SEP = 0, 1, 2, 3
    def __init__(self, max_seq_len=128):
        self.max_seq_len = max_seq_len
        self.word2id = {'[PAD]':0,'[UNK]':1,'[CLS]':2,'[SEP]':3}
        self.next_id = 4
    def _get_id(self, w):
        if w not in self.word2id:
            self.word2id[w] = self.next_id
            self.next_id += 1
        return self.word2id[w]
    def encode(self, text):
        tokens = [self.CLS] + [
            self._get_id(w.lower()) for w in text.split()
        ][:self.max_seq_len - 2] + [self.SEP]
        pad_len = self.max_seq_len - len(tokens)
        ids  = tokens + [self.PAD] * pad_len
        mask = [1]*len(tokens) + [0]*pad_len
        return {
            'input_ids'     : torch.tensor(ids,  dtype=torch.long),
            'attention_mask': torch.tensor(mask, dtype=torch.long),
        }
    @property
    def vocab_size(self):
        return max(30_522, self.next_id)

tokenizer = SimpleTokenizer(model_cfg.max_seq_len)
print('tokenizer ready')


## 3. 모델 아키텍첸

In [ ]:
class TokenEmbedding(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.token_emb = nn.Embedding(cfg.vocab_size, cfg.d_model, padding_idx=0)
        self.pos_emb   = nn.Embedding(cfg.max_seq_len, cfg.d_model)
        self.norm = nn.LayerNorm(cfg.d_model)
        self.drop = nn.Dropout(cfg.dropout)
        pos = torch.arange(cfg.max_seq_len).unsqueeze(1)
        dim = torch.arange(0, cfg.d_model, 2)
        pe  = torch.zeros(cfg.max_seq_len, cfg.d_model)
        pe[:, 0::2] = torch.sin(pos / 10000 ** (dim / cfg.d_model))
        pe[:, 1::2] = torch.cos(pos / 10000 ** (dim / cfg.d_model))
        self.pos_emb.weight.data.copy_(pe)
    def forward(self, ids):
        B, L = ids.shape
        pos = torch.arange(L, device=ids.device).unsqueeze(0)
        return self.drop(self.norm(self.token_emb(ids) + self.pos_emb(pos)))

class TransformerEncoderLayer(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.attn  = nn.MultiheadAttention(cfg.d_model, cfg.n_heads,
                                            dropout=cfg.dropout, batch_first=True)
        self.ff    = nn.Sequential(
            nn.Linear(cfg.d_model, cfg.d_ff), nn.GELU(),
            nn.Dropout(cfg.dropout),
            nn.Linear(cfg.d_ff, cfg.d_model),
        )
        self.norm1 = nn.LayerNorm(cfg.d_model)
        self.norm2 = nn.LayerNorm(cfg.d_model)
        self.drop  = nn.Dropout(cfg.dropout)
    def forward(self, x, kpm):
        r = x; x = self.norm1(x)
        x, _ = self.attn(x, x, x, key_padding_mask=kpm)
        x = self.drop(x) + r
        r = x; x = self.norm2(x)
        return self.drop(self.ff(x)) + r

class EmbeddingModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.embedding = TokenEmbedding(cfg)
        self.layers    = nn.ModuleList([TransformerEncoderLayer(cfg) for _ in range(cfg.n_layers)])
        self.proj      = nn.Sequential(
            nn.Linear(cfg.d_model, cfg.d_model), nn.GELU(),
            nn.Linear(cfg.d_model, cfg.embed_dim),
        )
    def forward(self, input_ids, attention_mask):
        kpm = (attention_mask == 0)
        x   = self.embedding(input_ids)
        for layer in self.layers:
            x = layer(x, kpm)
        mask = attention_mask.unsqueeze(-1).float()
        x = (x * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
        return F.normalize(self.proj(x), p=2, dim=-1)
    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

model = EmbeddingModel(model_cfg).to(train_cfg.device)
print(f'Parameters: {model.count_parameters():,}')
print(f'Embedding dim: {model_cfg.embed_dim}')


## 4. InfoNCE 손실 함수

In [ ]:
class InfoNCELoss(nn.Module):
    def __init__(self, temperature=0.05):
        super().__init__()
        self.temperature = temperature
    def forward(self, q_emb, p_emb):
        B = q_emb.size(0)
        sim    = torch.matmul(q_emb, p_emb.T) / self.temperature
        labels = torch.arange(B, device=q_emb.device)
        return (F.cross_entropy(sim, labels) + F.cross_entropy(sim.T, labels)) / 2

criterion = InfoNCELoss(train_cfg.temperature)
print(f'InfoNCE ready (temperature={train_cfg.temperature})')


## 5. 데이터셋

In [ ]:
class TripletDataset(Dataset):
    def __init__(self, pairs, tokenizer):
        self.pairs, self.tokenizer = pairs, tokenizer
    def __len__(self): return len(self.pairs)
    def __getitem__(self, idx):
        item = self.pairs[idx]
        q = self.tokenizer.encode(item['query'])
        p = self.tokenizer.encode(item['positive'])
        return {'q_input_ids': q['input_ids'], 'q_attention_mask': q['attention_mask'],
                'p_input_ids': p['input_ids'], 'p_attention_mask': p['attention_mask']}

def make_synthetic_pairs(n=8000):
    templates = [
        ('What is {concept}?',
         '{concept} is a core concept in {field} that involves {detail}.'),
        ('How does {concept} work?',
         'The mechanism of {concept} relies on {detail}, used widely in {field}.'),
        ('Why is {concept} important in {field}?',
         '{concept} is critical in {field} because it enables {detail}.'),
        ('Explain {concept} in {field}.',
         'In {field}, {concept} refers to {detail}, which is fundamental.'),
        ('What role does {concept} play in {field}?',
         '{concept} plays a key role in {field} by providing {detail}.'),
    ]
    concepts = [
        ('gradient descent',    'machine learning',    'minimizing loss via negative gradient steps'),
        ('attention mechanism', 'NLP',                 'computing weighted sums over token vectors'),
        ('photosynthesis',      'biology',             'converting sunlight and CO2 into glucose'),
        ('backpropagation',     'deep learning',       'computing gradients via chain rule'),
        ('CRISPR',              'genomics',            'targeted DNA editing with guide RNA'),
        ('Fourier transform',   'signal processing',   'decomposing signals into frequencies'),
        ('transformer model',   'deep learning',       'self-attention and feedforward stacking'),
        ('vector database',     'information retrieval','storing and searching embedding vectors'),
        ('RAG',                 'NLP',                 'augmenting LLMs with retrieved documents'),
        ('contrastive learning','self-supervised ML',  'pulling similar pairs close in embedding space'),
        ('mean pooling',        'NLP',                 'averaging token embeddings for a sentence vector'),
        ('cosine similarity',   'information retrieval','measuring angle between two vectors'),
        ('dropout',             'deep learning',       'randomly zeroing activations during training'),
        ('layer normalization', 'deep learning',       'normalizing activations within a layer'),
        ('beam search',         'NLP',                 'exploring top-k sequences at each decoding step'),
    ]
    pairs = []
    for _ in range(n):
        qt, pt = random.choice(templates)
        concept, field, detail = random.choice(concepts)
        pairs.append({
            'query'   : qt.format(concept=concept, field=field, detail=detail),
            'positive': pt.format(concept=concept, field=field, detail=detail),
        })
    return pairs

def load_msmarco(max_samples=50000):
    from datasets import load_dataset
    print(f'Loading MS MARCO (up to {max_samples:,})...')
    ds = load_dataset('ms_marco', 'v1.1', split='train', streaming=True)
    pairs = []
    for row in ds:
        for text, sel in zip(row['passages']['passage_text'], row['passages']['is_selected']):
            if sel == 1:
                pairs.append({'query': row['query'], 'positive': text})
                break
        if len(pairs) >= max_samples:
            break
    print(f'Loaded {len(pairs):,} pairs')
    return pairs

if USE_SYNTHETIC:
    print(f'Generating {SYNTHETIC_N:,} synthetic pairs...')
    all_pairs = make_synthetic_pairs(SYNTHETIC_N)
else:
    all_pairs = load_msmarco(MSMARCO_SAMPLES)

split = int(0.95 * len(all_pairs))
train_pairs, val_pairs = all_pairs[:split], all_pairs[split:]
print(f'Train: {len(train_pairs):,}  Val: {len(val_pairs):,}')

train_ds = TripletDataset(train_pairs, tokenizer)
val_ds   = TripletDataset(val_pairs,   tokenizer)
train_loader = DataLoader(train_ds, batch_size=train_cfg.batch_size,
                          shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=train_cfg.batch_size*2,
                          shuffle=False, num_workers=2)
print('DataLoaders ready')


## 6. 평가 함수 (MRR@10, Recall@K)

In [ ]:
@torch.no_grad()
def evaluate(model, val_loader, device):
    model.eval()
    all_q, all_p = [], []
    for batch in val_loader:
        q = model(batch['q_input_ids'].to(device), batch['q_attention_mask'].to(device))
        p = model(batch['p_input_ids'].to(device), batch['p_attention_mask'].to(device))
        all_q.append(q.cpu()); all_p.append(p.cpu())
    Q = torch.cat(all_q)
    P = torch.cat(all_p)
    sim   = torch.matmul(Q, P.T)
    ranks = (sim > sim.diagonal().unsqueeze(1)).sum(dim=1) + 1
    model.train()
    return {
        'MRR@10': (1.0 / ranks.float()).mean().item(),
        'R@1'   : (ranks <= 1).float().mean().item(),
        'R@5'   : (ranks <= 5).float().mean().item(),
        'R@10'  : (ranks <= 10).float().mean().item(),
    }

print('evaluate() ready')


## 7. 학습 실행 🚀

In [ ]:
device = train_cfg.device
optimizer = torch.optim.AdamW(model.parameters(),
                              lr=train_cfg.learning_rate, weight_decay=0.01)
total_steps = len(train_loader) * train_cfg.epochs

def lr_lambda(step):
    if step < train_cfg.warmup_steps:
        return step / max(1, train_cfg.warmup_steps)
    p = (step - train_cfg.warmup_steps) / max(1, total_steps - train_cfg.warmup_steps)
    return max(0.0, 0.5 * (1.0 + math.cos(math.pi * p)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler    = torch.cuda.amp.GradScaler() if device == 'cuda' else None

step, best_mrr = 0, 0.0
train_losses   = []

for epoch in range(1, train_cfg.epochs + 1):
    model.train()
    epoch_loss = 0.0
    pbar = tqdm(train_loader, desc=f'Epoch {epoch}/{train_cfg.epochs}')
    for batch in pbar:
        q_ids  = batch['q_input_ids'].to(device)
        q_mask = batch['q_attention_mask'].to(device)
        p_ids  = batch['p_input_ids'].to(device)
        p_mask = batch['p_attention_mask'].to(device)
        optimizer.zero_grad()
        if scaler:
            with torch.cuda.amp.autocast():
                loss = criterion(model(q_ids, q_mask), model(p_ids, p_mask))
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), train_cfg.max_grad_norm)
            scaler.step(optimizer); scaler.update()
        else:
            loss = criterion(model(q_ids, q_mask), model(p_ids, p_mask))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), train_cfg.max_grad_norm)
            optimizer.step()
        scheduler.step()
        step += 1
        epoch_loss += loss.item()
        train_losses.append(loss.item())
        pbar.set_postfix(loss=f'{loss.item():.4f}', lr=f'{scheduler.get_last_lr()[0]:.2e}')
        if step % train_cfg.eval_every == 0:
            m = evaluate(model, val_loader, device)
            print(f'Step {step:,} | ' + ' | '.join(f'{k}: {v:.4f}' for k,v in m.items()))
            if m['MRR@10'] > best_mrr:
                best_mrr = m['MRR@10']
                torch.save({'model_state': model.state_dict(), 'config': model_cfg},
                           train_cfg.save_path)
                print(f'  Best model saved (MRR@10={best_mrr:.4f})')
    print(f'Epoch {epoch} avg loss: {epoch_loss/len(train_loader):.4f}')

print(f'Training complete! Best MRR@10: {best_mrr:.4f}')


## 8. 학습 곡선 시각화

In [ ]:
import matplotlib.pyplot as plt

window = 50
smoothed = [
    sum(train_losses[max(0,i-window):i+1]) / len(train_losses[max(0,i-window):i+1])
    for i in range(len(train_losses))
]
plt.figure(figsize=(10, 4))
plt.plot(train_losses, alpha=0.3, color='steelblue', label='Raw loss')
plt.plot(smoothed,     color='steelblue',            label=f'{window}-step avg')
plt.xlabel('Step'); plt.ylabel('InfoNCE Loss')
plt.title('Training Loss Curve')
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()


## 9. 최종 평가

In [ ]:
ckpt = torch.load(train_cfg.save_path, map_location=device)
model.load_state_dict(ckpt['model_state'])
metrics = evaluate(model, val_loader, device)
print('Final validation results:')
for k, v in metrics.items():
    bar = chr(0x2588) * int(v * 40)
    print(f'  {k:8s}: {v:.4f}  {bar}')


## 10. 검색 데모 🔍

In [ ]:
class RAGRetriever:
    def __init__(self, model, tokenizer, device='cpu'):
        self.model, self.tokenizer, self.device = model.eval().to(device), tokenizer, device
        self._index, self._passages = None, []
    @torch.no_grad()
    def _embed(self, texts):
        embs = []
        for t in texts:
            enc = self.tokenizer.encode(t)
            e   = self.model(enc['input_ids'].unsqueeze(0).to(self.device),
                             enc['attention_mask'].unsqueeze(0).to(self.device))
            embs.append(e)
        return torch.cat(embs)
    def index(self, passages):
        self._passages = passages
        self._index    = self._embed(passages)
        print(f'{len(passages)} passages indexed')
    def search(self, query, top_k=3):
        scores = (self._embed([query]) @ self._index.T).squeeze(0)
        ids    = scores.topk(top_k).indices.tolist()
        return [{'rank':i+1, 'score':scores[j].item(), 'passage':self._passages[j]}
                for i,j in enumerate(ids)]

passages = [
    'Gradient descent minimizes a function by stepping in the negative gradient direction.',
    'Attention mechanisms compute weighted sums of value vectors based on query-key similarity.',
    'CRISPR-Cas9 enables precise editing of genomic DNA sequences using guide RNAs.',
    'RAG enhances LLMs by retrieving relevant documents at inference time.',
    'Contrastive learning trains models to pull similar pairs close in embedding space.',
    'Mean pooling averages all token embeddings to produce a fixed-size sentence vector.',
    'Cosine similarity measures the angle between two vectors regardless of magnitude.',
    'Backpropagation computes gradients through a neural network using the chain rule.',
    'Dropout randomly zeros activations during training to prevent overfitting.',
    'Layer normalization stabilizes training by normalizing activations within a layer.',
]

retriever = RAGRetriever(model, tokenizer, device=device)
retriever.index(passages)

for query in [
    'how does the attention mechanism work?',
    'what is retrieval augmented generation?',
    'explain gradient optimization',
]:
    print(f'Query: "{query}"')
    for r in retriever.search(query, top_k=3):
        print(f'  [{r["rank"]}] {r["score"]:.4f} | {r["passage"][:72]}...')
    print()


## 11. Google Drive 저장 (선택사항)

In [ ]:
# 세션 종료 후에도 모델을 유지하려면 Drive에 저장하세요.
# from google.colab import drive
# drive.mount('/content/drive')
# import shutil
# shutil.copy(train_cfg.save_path, '/content/drive/MyDrive/embedding_model.pt')
# print('Saved to Google Drive')
